### Rag pipline - Data Ingestion to Vector DB pipeline

In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [22]:
## Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):

    all_documents=[]
    pdf_dir= Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()


            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error : {e}")

    print(f"\n Total Documents loaded:{len(all_documents)}")
    return all_documents



Processed_documents= process_all_pdfs("../data/raw")

Found 2 PDF files to process

Processing:R22B.Tech.CSE(AIML)CourseStructureSyllabus2.pdf
Loaded 147 pages

Processing:R22B.Tech.RevisedAcademicRegulations2.pdf
Loaded 25 pages

 Total Documents loaded:172


all_documents

In [23]:
Processed_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2024-01-24T20:19:51+05:30', 'author': '', 'moddate': '2024-01-24T20:19:51+05:30', 'title': 'Microsoft Word - R22 B.Tech. CSE (AI&ML) Course Structure & Syllabus', 'source': '../data/raw/R22B.Tech.CSE(AIML)CourseStructureSyllabus2.pdf', 'total_pages': 147, 'page': 0, 'page_label': '1', 'source_file': 'R22B.Tech.CSE(AIML)CourseStructureSyllabus2.pdf', 'file_type': 'pdf'}, page_content='R22 B.Tech. CSE (AI and ML) Syllabus   JNTU Hyderabad \nPage 1 of 147 \n \nJAWAHARLAL NEHRU TECHNOLOGICAL UNIVERSITY HYDERABAD \nB.Tech. in COMPUTER SCIENCE AND ENGINEERING (AI & ML) \nCOURSE STRUCTURE & SYLLABUS (R22 Regulations) \nApplicable from AY 2022-23 Batch \n \nI YEAR I SEMESTER  \nS. \nNo. \nCourse \nCode \nCourse  L T  P Credits  \n1.  MA101BS Matrices and Calculus 3 1  0 4 \n2.  PH102BS Applied Physics 3 1  0 4 \n3.  CS103ES Programming for Problem Solving 3 0  0 3 \n4.  ME104ES Engineering Workshop 

In [25]:
## Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks ")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}.....")
        print(f"Metadata:{split_docs[0].metadata}")

    return split_docs


all_split_documents=split_documents(Processed_documents)
all_split_documents

split 172 documents into 532 chunks 

Example chunk:
Content: R22 B.Tech. CSE (AI and ML) Syllabus   JNTU Hyderabad 
Page 1 of 147 
 
JAWAHARLAL NEHRU TECHNOLOGICAL UNIVERSITY HYDERABAD 
B.Tech. in COMPUTER SCIENCE AND ENGINEERING (AI & ML) 
COURSE STRUCTURE & S.....
Metadata:{'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2024-01-24T20:19:51+05:30', 'author': '', 'moddate': '2024-01-24T20:19:51+05:30', 'title': 'Microsoft Word - R22 B.Tech. CSE (AI&ML) Course Structure & Syllabus', 'source': '../data/raw/R22B.Tech.CSE(AIML)CourseStructureSyllabus2.pdf', 'total_pages': 147, 'page': 0, 'page_label': '1', 'source_file': 'R22B.Tech.CSE(AIML)CourseStructureSyllabus2.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2024-01-24T20:19:51+05:30', 'author': '', 'moddate': '2024-01-24T20:19:51+05:30', 'title': 'Microsoft Word - R22 B.Tech. CSE (AI&ML) Course Structure & Syllabus', 'source': '../data/raw/R22B.Tech.CSE(AIML)CourseStructureSyllabus2.pdf', 'total_pages': 147, 'page': 0, 'page_label': '1', 'source_file': 'R22B.Tech.CSE(AIML)CourseStructureSyllabus2.pdf', 'file_type': 'pdf'}, page_content='R22 B.Tech. CSE (AI and ML) Syllabus   JNTU Hyderabad \nPage 1 of 147 \n \nJAWAHARLAL NEHRU TECHNOLOGICAL UNIVERSITY HYDERABAD \nB.Tech. in COMPUTER SCIENCE AND ENGINEERING (AI & ML) \nCOURSE STRUCTURE & SYLLABUS (R22 Regulations) \nApplicable from AY 2022-23 Batch \n \nI YEAR I SEMESTER  \nS. \nNo. \nCourse \nCode \nCourse  L T  P Credits  \n1.  MA101BS Matrices and Calculus 3 1  0 4 \n2.  PH102BS Applied Physics 3 1  0 4 \n3.  CS103ES Programming for Problem Solving 3 0  0 3 \n4.  ME104ES Engineering Workshop 

# Embedding and vector Db

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Setiings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name= Huggingface model name for sentence embeddings
        """
        self.model_name=model_name
        self.mode=model
        self._load_model()
    
    def _load_mode(self):
        """ Load the sentenceTransformer model"""

        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding Dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.model_name}:{e}")
            raise
    
    def generate_embedding(self,texts:List[str])->np.ndarray:
